# Phase 3: Logistic Regression Scorecard Model
In this notebook, we build the champion credit scorecard. We train a Logistic Regression model on the selected WoE features, check model coefficients and multicollinearity (VIF), and scale log-odds into score points.


In [ ]:
import pandas as pd
import os
import sys
import yaml

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from woe_binning import WoEBinning
from scorecard import ScorecardModel


## 1. Load Data & Mappings
Load the dataset splits and the WoE mapping object.


In [ ]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()

woe_model = WoEBinning(target_col='target')
woe_model.fit_all(train_df, [
    'loan_amnt', 'annual_inc', 'dti', 'revol_util', 
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 
    'pub_rec', 'pub_rec_bankruptcies', 'credit_history_age',
    'emp_length'
], ['home_ownership', 'purpose'])

train_woe = woe_model.transform(train_df)
selected_woe_cols = [f'{col}_woe' for col in woe_model.selected_features]


## 2. Train Logistic Regression
We train the regression model predicting default (target=1) using statsmodels.


In [ ]:
scorecard = ScorecardModel(base_score=600, base_odds=50, pdo=20)
scorecard.fit(train_woe[selected_woe_cols], train_woe['target'])
print(scorecard.model.summary())


## 3. Multicollinearity & Diagnostic Checks
We check Variance Inflation Factors (VIF) and verify that all coefficients are statistically significant (p < 0.05) and have the expected sign (negative for default prediction).


In [ ]:
print('Variance Inflation Factor (VIF) Results:')
print(scorecard.vif_report)

is_valid, issues = scorecard.validate_coefficients()
print('\nModel Diagnostics Check:')
if is_valid:
    print('SUCCESS: All coefficient signs are correct and VIF values are within regulatory limits.')
else:
    print('WARNING Issues Detected:')
    for issue in issues:
        print(' -', issue)


## 4. Scorecard Scaling & Points Mapping
Translate the logistic regression coefficients into integer-based score contributions per bin.


In [ ]:
scorecard.build_scorecard_table(woe_model.mappings)
scorecard_table = scorecard.scorecard_table
print('Scorecard Points Table (First 15 Rows):')
print(scorecard_table.head(15))


## 5. Customer Score Generation
Demonstrate credit scoring on train sample and show the output scores and PD values.


In [ ]:
scores = scorecard.predict_score(train_woe)
pds = scorecard.predict_pd(train_woe)
scored_df = pd.DataFrame({'Target': train_woe['target'], 'Credit Score': scores, 'PD': pds})
print(scored_df.describe())
